# ArcFace Palmprint Verification: End-to-End Training & Evaluation Notebook
### Dataset: Tongji Contactless Palmprint Dataset
### Architecture: ResNet-18 Backbone + 512-D PReLU Projection + Additive Angular Margin Loss (ArcFace)

---
### 🔧 FIX LOG (v3) - PERFORMANCE BOOST
- **Removed Zhang ROI:** Since images are already pre-cropped (128x128), Zhang was destroying the features.
- **Added CLAHE:** Enhances the palm lines for better feature extraction.
- **Fixed Validation Bug:** Removed hardcoded 0.62 threshold during training to stop accuracy from dropping.
- **Optimized Augmentations:** Reduced random rotations to prevent distorting the pre-cropped images.

## Cell 1: Environment & Hardware Acceleration Setup

In [ ]:
# =============================================================================
# Cell 1: Environment, Dependencies & Hardware Configuration
# =============================================================================
import os, sys, math, random, re, time, hashlib, json
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Union

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import auc, roc_curve
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm import tqdm

IS_KAGGLE = os.path.exists('/kaggle')
INPUT_DIR = '/kaggle/input' if IS_KAGGLE else 'data/raw'
WORKING_DIR = '/kaggle/working' if IS_KAGGLE else 'outputs_kaggle'

CKPT_DIR = os.path.join(WORKING_DIR, 'checkpoints')
OUT_DIR = os.path.join(WORKING_DIR, 'outputs')
LOGS_DIR = os.path.join(WORKING_DIR, 'logs')
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda:0')
    gpu_name = torch.cuda.get_device_name(0)
    print(f'[HARDWARE] GPU Active: {gpu_name}')
    torch.backends.cudnn.benchmark = True
else:
    DEVICE = torch.device('cpu')
    print('[HARDWARE] Running on CPU')

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(42)
print(f'[SETUP] Seed fixed to 42. Outputs will be saved to: {WORKING_DIR}')

## Cell 2: Dataset Discovery & Person-Level Partitioning

In [ ]:
# =============================================================================
# Cell 2: Dataset Discovery, Structure Parser & Person Split
# =============================================================================
@dataclass
class PalmSample:
    file_path: str
    person_id: str
    session_id: str
    sample_index: int

def find_tongji_dataset_root(search_base: str) -> str:
    if not os.path.exists(search_base):
        return 'data/raw/tongji'
    for root, dirs, files in os.walk(search_base):
        has_bmps = any(f.lower().endswith(('.bmp', '.jpg', '.png')) for f in files)
        has_sessions = any(d.lower() in ['session1', 'session2', '001', 'session_1'] for d in dirs)
        if has_bmps and has_sessions:
            return root
        if has_sessions:
            return root
    return search_base

def parse_tongji_dataset(root_dir: str) -> List[PalmSample]:
    valid_exts = {'.bmp', '.jpg', '.jpeg', '.png'}
    samples = []
    root = Path(root_dir)
    if not root.exists():
        return samples

    session_dirs = [d for d in root.iterdir() if d.is_dir() and 'session' in d.name.lower()]
    if session_dirs:
        for s_dir in session_dirs:
            sess_id = 'session2' if '2' in s_dir.name.lower() else 'session1'
            for f in sorted(s_dir.iterdir()):
                if f.suffix.lower() in valid_exts:
                    m = re.search(r'(\d+)[_-](\d+)', f.stem)
                    p_id = m.group(1).zfill(3) if m else f.stem.split('_')[0].zfill(3)
                    idx = int(m.group(2)) if m else 0
                    samples.append(PalmSample(str(f), p_id, sess_id, idx))
    else:
        for p_dir in sorted(root.iterdir()):
            if not p_dir.is_dir() or p_dir.name.startswith('.'):
                continue
            p_id = p_dir.name.zfill(3)
            for f in sorted(p_dir.glob('**/*')):
                if f.is_file() and f.suffix.lower() in valid_exts:
                    sess_id = 'session2' if ('2' in f.name.lower() or 's2' in f.name.lower()) else 'session1'
                    samples.append(PalmSample(str(f), p_id, sess_id, 0))
    return samples

def split_dataset_by_person(samples: List[PalmSample], seed: int = 42):
    rng = random.Random(seed)
    person_ids = sorted(list(set(s.person_id for s in samples)))
    rng.shuffle(person_ids)
    
    n_train = max(1, int(round(len(person_ids) * 0.70)))
    n_val = max(1, int(round(len(person_ids) * 0.15)))
    
    train_ids = set(person_ids[:n_train])
    val_ids = set(person_ids[n_train : n_train + n_val])
    test_ids = set(person_ids[n_train + n_val :])
    if len(test_ids) == 0 and len(val_ids) > 0:
        test_ids = val_ids
    
    train_s = [s for s in samples if s.person_id in train_ids]
    val_s = [s for s in samples if s.person_id in val_ids]
    test_s = [s for s in samples if s.person_id in test_ids]
    
    return train_s, val_s, test_s, len(train_ids)

DATA_ROOT = find_tongji_dataset_root(INPUT_DIR)
print(f'[DATASET SCAN] Searching in: {DATA_ROOT}')
ALL_SAMPLES = parse_tongji_dataset(DATA_ROOT)
print(f'[FOUND] {len(ALL_SAMPLES)} total palmprint images.')

## Cell 3: Instant Dry-Run Synthetic Dataset (Zero-Crash Fallback)

In [ ]:
# =============================================================================
# Cell 3: Synthetic Palm Generator & Active Partitioning
# =============================================================================
def generate_synthetic_tongji_demo(demo_root: str, num_persons: int = 10, per_session: int = 4):
    os.makedirs(demo_root, exist_ok=True)
    rng = np.random.default_rng(42)
    created_samples = []
    print(f'[SYNTHESIS] Generating {num_persons} synthetic palm identities in {demo_root}...')
    for p_idx in range(1, num_persons + 1):
        pid = f'{p_idx:03d}'
        base_heart_y = int(120 + rng.uniform(-15, 15))
        base_head_y = int(160 + rng.uniform(-15, 15))
        base_life_x = int(90 + rng.uniform(-10, 10))
        
        for sess_id in ['session1', 'session2']:
            s_dir = os.path.join(demo_root, sess_id)
            os.makedirs(s_dir, exist_ok=True)
            for img_idx in range(1, per_session + 1):
                canvas = np.full((256, 256), 28, dtype=np.uint8)
                cv2.ellipse(canvas, (128, 140), (95, 105), 0, 0, 360, 185, -1)
                jitter = rng.integers(-3, 4)
                cv2.ellipse(canvas, (130, base_heart_y + jitter), (70, 35), -15, 180, 360, 70, 3)
                cv2.ellipse(canvas, (120, base_head_y + jitter), (65, 30), -5, 180, 360, 75, 3)
                cv2.ellipse(canvas, (base_life_x + jitter, 160), (35, 75), 10, 90, 270, 80, 3)
                noise = rng.normal(0, 10, canvas.shape).astype(np.int16)
                noisy = np.clip(canvas.astype(np.int16) + noise, 0, 255).astype(np.uint8)
                blurred = cv2.GaussianBlur(noisy, (3, 3), 1.0)
                file_path = os.path.join(s_dir, f'{pid}_{img_idx}.png')
                cv2.imwrite(file_path, blurred)
                created_samples.append(PalmSample(file_path, pid, sess_id, img_idx))
    return created_samples

if len(ALL_SAMPLES) == 0:
    print('[NOTICE] External Tongji dataset not detected.')
    print('[ACTION] Activating built-in synthetic biometric generator...')
    DEMO_ROOT = os.path.join(WORKING_DIR, 'synthetic_tongji_demo')
    ALL_SAMPLES = generate_synthetic_tongji_demo(DEMO_ROOT, num_persons=12, per_session=4)

TRAIN_SAMPLES, VAL_SAMPLES, TEST_SAMPLES, NUM_TRAIN_CLASSES = split_dataset_by_person(ALL_SAMPLES, seed=42)
print(f'[SPLIT READY] Train: {len(TRAIN_SAMPLES)} imgs ({NUM_TRAIN_CLASSES} identities) | Val: {len(VAL_SAMPLES)} imgs | Test: {len(TEST_SAMPLES)} imgs')

## Cell 4: Image Preprocessing (CLAHE + Resize) & Data Augmentations

### 🔧 MAJOR FIX APPLIED
1. Removed Zhang ROI (since images are already pre-cropped 128x128).
2. Added **CLAHE** (Contrast Limited Adaptive Histogram Equalization) to make palm lines sharper.
3. Reduced aggressive rotations.

In [ ]:
# =============================================================================
# Cell 4: Preprocessing (CLAHE + Resize) & Data Augmentations
# =============================================================================
def robust_roi(gray: np.ndarray, target_size: int = 224) -> np.ndarray:
    """
    Since images are already perfectly cropped (e.g., 128x128), we bypass Zhang.
    We apply CLAHE to enhance the palmprint ridges/lines.
    """
    # Apply CLAHE for contrast enhancement
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    enhanced = clahe.apply(gray)
    
    # Directly resize to 224x224
    return cv2.resize(enhanced, (target_size, target_size), interpolation=cv2.INTER_CUBIC)

class RGBReplicate:
    def __call__(self, img: Image.Image) -> Image.Image:
        if img.mode != 'L':
            img = img.convert('L')
        return Image.merge('RGB', (img, img, img))

TRAIN_TRANSFORMS = transforms.Compose([
    RGBReplicate(),
    transforms.RandomResizedCrop(224, scale=(0.95, 1.0), ratio=(0.98, 1.02)), # Very slight crop
    transforms.RandomRotation(5.0), # Reduced from 10.0
    transforms.RandomAffine(0, translate=(0.01, 0.01), shear=2.0),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    transforms.RandomErasing(p=0.15, scale=(0.02, 0.08), value=0.0),
])

VAL_TRANSFORMS = transforms.Compose([
    RGBReplicate(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])
print('[PREPROCESSING READY] CLAHE + Resize configured. Zhang ROI removed for pre-cropped images.')

## Cell 5: Model Architecture & ArcFace Additive Margin Head

In [ ]:
# =============================================================================
# Cell 5: ResNet-18 Backbone, 512-D Projection & ArcFace Head
# =============================================================================
class ArcFaceHead(nn.Module):
    def __init__(self, embedding_dim: int = 512, num_classes: int = 210, margin: float = 0.35, scale: float = 30.0):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(num_classes, embedding_dim))
        nn.init.xavier_uniform_(self.weight)
        self.margin = margin
        self.scale = scale
        self.cos_m = math.cos(margin)
        self.sin_m = math.sin(margin)
        self.th = math.cos(math.pi - margin)
        self.mm = math.sin(math.pi - margin) * margin

    def forward(self, embeddings: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        with torch.amp.autocast(device_type=embeddings.device.type, enabled=False):
            embeddings = embeddings.float()
            W = F.normalize(self.weight.float(), p=2, dim=1)
            cosine = F.linear(embeddings, W).clamp(-1.0 + 1e-7, 1.0 - 1e-7)
            sine = torch.sqrt((1.0 - cosine ** 2).clamp(0.0, 1.0))
            phi = torch.where(cosine > self.th, cosine * self.cos_m - sine * self.sin_m, cosine - self.mm)
            one_hot = F.one_hot(labels, num_classes=self.weight.size(0)).float()
            logits = (one_hot * phi + (1.0 - one_hot) * cosine) * self.scale
        return logits

class PalmprintArcFace(nn.Module):
    def __init__(self, num_classes: int = 210, embedding_dim: int = 512, pretrained: bool = True):
        super().__init__()
        base = models.resnet18(weights='DEFAULT' if pretrained else None)
        in_features = base.fc.in_features
        base.fc = nn.Identity()
        self.backbone = base

        self.embedding_head = nn.Sequential(
            nn.Linear(in_features, embedding_dim, bias=False),
            nn.BatchNorm1d(embedding_dim),
            nn.PReLU(init=0.25),
            nn.Dropout(0.4), # Increased dropout slightly
            nn.Linear(embedding_dim, embedding_dim, bias=False),
            nn.BatchNorm1d(embedding_dim),
        )
        self.arcface = ArcFaceHead(embedding_dim=embedding_dim, num_classes=num_classes)

    def extract_embedding(self, x: torch.Tensor) -> torch.Tensor:
        feat = self.backbone(x)
        emb = self.embedding_head(feat)
        return F.normalize(emb, p=2, dim=1)

    def forward(self, x: torch.Tensor, labels: Optional[torch.Tensor] = None):
        emb = self.extract_embedding(x)
        if labels is not None:
            return self.arcface(emb, labels), emb
        return emb

test_net = PalmprintArcFace(num_classes=NUM_TRAIN_CLASSES, pretrained=False).to(DEVICE)
test_in = torch.randn(2, 3, 224, 224, device=DEVICE)
test_emb = test_net.extract_embedding(test_in)
print(f'[ARCHITECTURE VERIFIED] Output Embedding Shape: {test_emb.shape}')

## Cell 6: Dataset Loaders & Cross-Session Pair Generation

In [ ]:
# =============================================================================
# Cell 6: PyTorch Dataset & Cross-Session Pair Generator
# =============================================================================
class PalmprintClassificationDataset(Dataset):
    def __init__(self, samples: List[PalmSample], transform=None):
        self.samples = samples
        unique_ids = sorted(list(set(s.person_id for s in samples)))
        self.p2idx = {p: i for i, p in enumerate(unique_ids)}
        self.transform = transform or TRAIN_TRANSFORMS

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        with Image.open(s.file_path) as img:
            gray = np.array(img.convert('L'))
        roi = robust_roi(gray)
        tensor = self.transform(Image.fromarray(roi, mode='L'))
        return tensor, self.p2idx[s.person_id]

class PalmprintPairDataset(Dataset):
    def __init__(self, samples: List[PalmSample], num_pairs: int = 1500, seed: int = 42):
        self.samples = samples
        self.transform = VAL_TRANSFORMS
        self.pairs = self._make_pairs(samples, num_pairs, seed)

    def _make_pairs(self, samples, num_pairs, seed):
        rng = random.Random(seed)
        pdict = {}
        for s in samples:
            pdict.setdefault(s.person_id, {}).setdefault(s.session_id, []).append(s)
        pos, neg = [], []
        for pid, sdict in pdict.items():
            s1, s2 = sdict.get('session1', []), sdict.get('session2', [])
            for a in s1:
                for b in s2:
                    pos.append((a, b, 1))
        rng.shuffle(pos)
        pos = pos[: max(1, num_pairs // 2)]
        while len(neg) < len(pos) and len(samples) >= 2:
            a, b = rng.choice(samples), rng.choice(samples)
            if a.person_id != b.person_id:
                neg.append((a, b, 0))
        all_pairs = pos + neg
        rng.shuffle(all_pairs)
        return all_pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        s1, s2, label = self.pairs[idx]
        with Image.open(s1.file_path) as p1:
            g1 = np.array(p1.convert('L'))
        with Image.open(s2.file_path) as p2:
            g2 = np.array(p2.convert('L'))
        r1 = robust_roi(g1)
        r2 = robust_roi(g2)
        return self.transform(Image.fromarray(r1, mode='L')), self.transform(Image.fromarray(r2, mode='L')), label

print(f'[DATASETS CONFIGURED] Ready for training and validation.')

## Cell 7: Mixed Precision Training Loop

### 🔧 BUG FIXED
The hardcoded `0.62` threshold in validation is removed. We now calculate optimal EER dynamically at each validation step.

In [ ]:
# =============================================================================
# Cell 7: Full Training Engine with Dynamic EER Validation
# =============================================================================
def compute_eer(scores, labels):
    sweep_th = np.linspace(-1.0, 1.0, 1000)
    far_list, frr_list = [], []
    pos_s = scores[labels == 1]
    neg_s = scores[labels == 0]
    for th in sweep_th:
        far_list.append(np.sum(neg_s >= th) / max(1, len(neg_s)))
        frr_list.append(np.sum(pos_s < th) / max(1, len(pos_s)))
    far_arr, frr_arr = np.array(far_list), np.array(frr_list)
    eer_idx = np.argmin(np.abs(far_arr - frr_arr))
    eer = float((far_arr[eer_idx] + frr_arr[eer_idx]) / 2.0)
    eer_th = float(sweep_th[eer_idx])
    return eer, eer_th

def run_training_experiment(epochs: int = 30, batch_size: int = 32):
    train_ds = PalmprintClassificationDataset(TRAIN_SAMPLES)
    val_ds = PalmprintPairDataset(VAL_SAMPLES, num_pairs=1000)

    num_workers = 2 if os.name != 'nt' else 0
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=(DEVICE.type=='cuda'), drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)

    model = PalmprintArcFace(num_classes=NUM_TRAIN_CLASSES, pretrained=True).to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    
    # Adjusted Learning Rates and Weight Decay
    optimizer = AdamW([
        {'params': model.backbone.parameters(), 'lr': 5e-5}, 
        {'params': model.embedding_head.parameters(), 'lr': 1e-4},
        {'params': model.arcface.parameters(), 'lr': 1e-4},
    ], weight_decay=1e-3)

    warmup_epochs = min(3, max(1, epochs // 5))
    warmup = LinearLR(optimizer, start_factor=0.1, total_iters=warmup_epochs)
    cosine = CosineAnnealingLR(optimizer, T_max=max(1, epochs - warmup_epochs), eta_min=1e-6)
    scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[warmup_epochs])
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=(DEVICE.type == 'cuda'))

    best_eer = 1.0
    best_ckpt_path = os.path.join(CKPT_DIR, 'best_model.pth')

    print(f'Starting training for {epochs} epochs on {DEVICE}...')
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        pbar = tqdm(train_loader, desc=f'Epoch {epoch:02d}/{epochs}', leave=False)
        for imgs, lbls in pbar:
            imgs, lbls = imgs.to(DEVICE, non_blocking=True), lbls.to(DEVICE, non_blocking=True)
            optimizer.zero_grad()
            with torch.amp.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == 'cuda')):
                logits, _ = model(imgs, lbls)
                loss = criterion(logits, lbls)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item() * imgs.size(0)
            correct += (torch.argmax(logits, 1) == lbls).sum().item()
            total += imgs.size(0)
            pbar.set_postfix({'loss': f'{(total_loss/max(1,total)):.4f}'})

        scheduler.step()

        # Validation every 2 epochs
        if epoch % 2 == 0 or epoch == epochs:
            model.eval()
            scores, labels = [], []
            with torch.no_grad():
                for i1, i2, l in val_loader:
                    e1 = model.extract_embedding(i1.to(DEVICE))
                    e2 = model.extract_embedding(i2.to(DEVICE))
                    scores.extend(torch.sum(e1 * e2, dim=1).cpu().numpy().tolist())
                    labels.extend(l.numpy().tolist())
            
            if len(scores) > 0:
                scores, labels = np.array(scores), np.array(labels)
                eer, eer_th = compute_eer(scores, labels)
                print(f'--> Epoch {epoch:02d} | Loss: {(total_loss/max(1,total)):.4f} | Val EER: {eer*100:.2f}% (Best Th: {eer_th:.3f})')
                
                # Save model with LOWEST EER (Higher is better accuracy)
                if eer < best_eer:
                    best_eer = eer
                    torch.save(model.state_dict(), best_ckpt_path)
                    print(f'*** Saved best model (EER: {eer*100:.2f}%) to {best_ckpt_path} ***')

run_training_experiment(epochs=30, batch_size=32)

## Cell 8: Biometric Evaluation Suite (ROC Curve, EER & Thresholds)

In [ ]:
# =============================================================================
# Cell 8: Scientific Biometric Evaluation & ROC Curve Plotting
# =============================================================================
def evaluate_test_set():
    test_ds = PalmprintPairDataset(TEST_SAMPLES, num_pairs=1000)
    test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)
    model = PalmprintArcFace(num_classes=NUM_TRAIN_CLASSES, pretrained=False).to(DEVICE)
    best_ckpt = os.path.join(CKPT_DIR, 'best_model.pth')
    if os.path.exists(best_ckpt):
        model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
        print(f'Loaded weights from: {best_ckpt}')
    model.eval()

    scores, labels = [], []
    with torch.no_grad():
        for i1, i2, l in tqdm(test_loader, desc='Test Pair Evaluation'):
            e1 = model.extract_embedding(i1.to(DEVICE))
            e2 = model.extract_embedding(i2.to(DEVICE))
            scores.extend(torch.sum(e1 * e2, dim=1).cpu().numpy().tolist())
            labels.extend(l.numpy().tolist())

    scores, labels = np.array(scores), np.array(labels)
    fpr, tpr, _ = roc_curve(labels, scores)
    roc_auc = float(auc(fpr, tpr))
    eer, eer_th = compute_eer(scores, labels)

    plt.figure(figsize=(7, 5), dpi=150)
    plt.plot(fpr, tpr, color='#06B6D4', lw=2.2, label=f'ArcFace ResNet-18 (AUC = {roc_auc:.4f})')
    plt.plot([0, 1], [0, 1], color='#64748B', linestyle='--')
    plt.scatter([eer], [1.0 - eer], color='#EF4444', s=45, label=f'EER = {eer*100:.2f}% @ Th={eer_th:.3f}')
    plt.xscale('log')
    plt.xlim([1e-4, 1.0])
    plt.ylim([0.70, 1.005])
    plt.xlabel('False Acceptance Rate (FAR, log scale)')
    plt.ylabel('True Acceptance Rate (TAR)')
    plt.title('ROC Curve - Tongji Palmprint Verification (CLAHE Enhanced)')
    plt.grid(True, which='both', linestyle=':', alpha=0.5)
    plt.legend(loc='lower right')
    plt.tight_layout()
    roc_save_path = os.path.join(OUT_DIR, 'roc_curve.png')
    plt.savefig(roc_save_path)
    plt.show()

    thresholds = {'eer_threshold': round(eer_th, 4), 'default': round(eer_th, 4)}
    with open(os.path.join(OUT_DIR, 'threshold.json'), 'w') as f:
        json.dump(thresholds, f, indent=2)

    print('=============================================')
    print('EVALUATION RESULTS:')
    print(f'EER: {eer*100:.2f}% | AUC: {roc_auc:.4f} | Optimal Threshold: {eer_th:.4f}')
    print(f'Artifacts saved in: {OUT_DIR}')
    print('=============================================')

evaluate_test_set()

## Cell 9: 1:1 Live Palmprint Verification Inference

In [ ]:
# =============================================================================
# Cell 9: 1:1 Verification Function
# =============================================================================
def verify_two_palms(img1_path: str, img2_path: str, threshold: float = 0.62):
    # Try to load the threshold from Cell 8 if it exists
    th_path = os.path.join(OUT_DIR, 'threshold.json')
    if os.path.exists(th_path):
        with open(th_path, 'r') as f:
            threshold = json.load(f)['eer_threshold']

    model = PalmprintArcFace(num_classes=NUM_TRAIN_CLASSES, pretrained=False).to(DEVICE)
    best_ckpt = os.path.join(CKPT_DIR, 'best_model.pth')
    if os.path.exists(best_ckpt):
        model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
    model.eval()

    with Image.open(img1_path) as p1:
        g1 = np.array(p1.convert('L'))
    with Image.open(img2_path) as p2:
        g2 = np.array(p2.convert('L'))
    
    r1 = robust_roi(g1)
    r2 = robust_roi(g2)

    t1 = VAL_TRANSFORMS(Image.fromarray(r1, mode='L')).unsqueeze(0).to(DEVICE)
    t2 = VAL_TRANSFORMS(Image.fromarray(r2, mode='L')).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        e1 = model.extract_embedding(t1).squeeze(0).cpu().numpy()
        e2 = model.extract_embedding(t2).squeeze(0).cpu().numpy()

    cos_sim = float(np.dot(e1, e2))
    dist = float(np.linalg.norm(e1 - e2))
    is_match = cos_sim >= threshold

    print('========================================')
    print('PALMPRINT VERIFICATION RESULT')
    print('========================================')
    print(f'Image 1:            {img1_path}')
    print(f'Image 2:            {img2_path}')
    print(f'Decision:           {"SAME PERSON (MATCH)" if is_match else "DIFFERENT PERSON (IMPOSTOR)"}')
    print(f'Cosine Similarity:  {cos_sim:.4f}')
    print(f'Euclidean Distance: {dist:.4f}')
    print(f'Decision Threshold: {threshold:.4f} (Optimized)')
    print(f'Confidence Margin:  {abs(cos_sim - threshold):.4f}')
    print('========================================')
    return {'match': is_match, 'cosine': cos_sim, 'distance': dist}

if len(TEST_SAMPLES) >= 2:
    s1, s2 = TEST_SAMPLES[0], TEST_SAMPLES[1]
    verify_two_palms(s1.file_path, s2.file_path)
print('[COMPLETE] ArcFace Palmprint Pipeline ready for full production use!')